# 04 · V-JEPA 2.1-B + dense CAN head smoke test

Verifies model loading, shapes, GPU memory, forward pass and one backward pass before a long run.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

# ============================================================
# Repository
# ============================================================
REPO = Path("/content/Blackbox-Detection")
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage3-sangchun"

if not REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "--branch", BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO), "fetch", "origin", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO), "checkout", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )

# ============================================================
# Paths
# ============================================================
DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATA_ROOT = DRIVE_ROOT / "DATASET"

COMMA_ROOT = DATA_ROOT / "comma2k19"
RAW_ROOT = COMMA_ROOT / "raw"
PROCESSED_ROOT = COMMA_ROOT / "processed" / "v1"

MANIFEST_ROOT = DRIVE_ROOT / "manifests" / "stage3" / "v1"
OUTPUT_ROOT = DRIVE_ROOT / "outputs" / "stage3"
PRETRAINED_ROOT = DRIVE_ROOT / "pretrained"

for p in [PROCESSED_ROOT, MANIFEST_ROOT, OUTPUT_ROOT, PRETRAINED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# ============================================================
# Install project from the existing pyproject.toml
# --no-deps keeps Colab's/DACON's binary stack intact.
# ============================================================
subprocess.run(
    [
        sys.executable,
        "-m", "pip", "install",
        "-q", "--no-deps", "-e", str(REPO),
    ],
    check=True,
)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()

print("Repository     :", REPO)
print("Branch         :", BRANCH)
print("Commit         :", commit)
print("RAW_ROOT       :", RAW_ROOT)
print("PROCESSED_ROOT :", PROCESSED_ROOT)

Mounted at /content/drive
Repository     : /content/Blackbox-Detection
Branch         : stage3-sangchun
Commit         : ec5eb78
RAW_ROOT       : /content/drive/MyDrive/Blackbox-Detection/DATASET/comma2k19/raw
PROCESSED_ROOT : /content/drive/MyDrive/Blackbox-Detection/DATASET/comma2k19/processed/v1


In [2]:
VJEPA_REPO = Path("/content/vjepa2")
VJEPA_COMMIT = "45d025f636dfc58fc2426905fc4a1ab755b1c3e5"

if not VJEPA_REPO.exists():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/facebookresearch/vjepa2.git", str(VJEPA_REPO)],
        check=True,
    )

subprocess.run(
    ["git", "-C", str(VJEPA_REPO), "fetch", "--all", "--tags"],
    check=True,
)
subprocess.run(
    ["git", "-C", str(VJEPA_REPO), "checkout", "-q", VJEPA_COMMIT],
    check=True,
)

print(
    "V-JEPA commit:",
    subprocess.check_output(
        ["git", "-C", str(VJEPA_REPO), "rev-parse", "HEAD"],
        text=True,
    ).strip(),
)

VJEPA_CKPT = PRETRAINED_ROOT / "vjepa2_1_vitb_dist_vitG_384.pt"
if not VJEPA_CKPT.exists():
    subprocess.run(
        [
            "wget", "-q",
            "-O", str(VJEPA_CKPT),
            "https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt",
        ],
        check=True,
    )

print(VJEPA_CKPT, f"{VJEPA_CKPT.stat().st_size / 2**20:.1f} MiB")

V-JEPA commit: 45d025f636dfc58fc2426905fc4a1ab755b1c3e5
/content/drive/MyDrive/Blackbox-Detection/pretrained/vjepa2_1_vitb_dist_vitG_384.pt 1587.1 MiB


In [3]:
import json
import torch
from torch.utils.data import DataLoader

from blackbox_detection.stage3.dataset import Stage3CANDataset
from blackbox_detection.stage3.vjepa21 import load_vjepa21_base_encoder
from blackbox_detection.stage3.models import VJEPA21DenseCAN
from blackbox_detection.stage3.losses import can_multitask_loss

stats_path = MANIFEST_ROOT / "target_stats.json"
train_manifest = MANIFEST_ROOT / "comma_train.csv"

stats = json.loads(stats_path.read_text())

ds = Stage3CANDataset(
    train_manifest,
    PROCESSED_ROOT,
    target_stats=stats,
    clip_len=16,
    window_stride=16,
    input_size=(288, 384),
    random_flip=True,
    max_windows=16,
)

loader = DataLoader(
    ds,
    batch_size=1,
    shuffle=False,
    num_workers=0,
)

batch = next(iter(loader))
print("video :", batch["video"].shape, batch["video"].dtype)
print("target:", batch["target"].shape)

video : torch.Size([1, 3, 16, 288, 384]) torch.float32
target: torch.Size([1, 16, 4])


In [4]:
assert torch.cuda.is_available(), "Use a Colab GPU runtime (L4)."

device = torch.device("cuda")
torch.cuda.reset_peak_memory_stats()

backbone = load_vjepa21_base_encoder(
    VJEPA_REPO,
    VJEPA_CKPT,
    num_frames=16,
    out_layers=(2, 5, 8, 11),
    freeze=True,
)

model = VJEPA21DenseCAN(
    backbone,
    freeze_backbone=True,
).to(device)

video = batch["video"].to(device)
target = batch["target"].to(device)
valid = batch["valid"].to(device)

with torch.autocast("cuda", dtype=torch.bfloat16):
    out = model(video)
    loss, parts = can_multitask_loss(out, target, valid)

print({k: v.shape for k, v in out.items()})
print("loss:", loss.item(), parts)

loss.backward()

print(
    "max allocated GiB:",
    torch.cuda.max_memory_allocated() / 2**30,
)

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


{'speed_mps': torch.Size([1, 16]), 'accel_from_speed_mps2': torch.Size([1, 16]), 'steering_deg': torch.Size([1, 16]), 'yaw_rate_rps': torch.Size([1, 16])}
loss: 0.5500888824462891 {'speed_mps': 0.49298790097236633, 'accel_from_speed_mps2': 0.03228096663951874, 'steering_deg': 0.009682971984148026, 'yaw_rate_rps': 0.01513708382844925, 'total': 0.5500888824462891}
max allocated GiB: 0.4919734001159668


If this fits comfortably on the L4, notebook 05 can start with batch size 2.
If memory is tight, keep batch size 1 and increase gradient accumulation.